# Spacecraft Anomaly Triage — Colab LSTM Benchmark

Runs the full LSTM-on-real-NASA-data benchmark on a Colab T4 GPU.

**Before you Run All:**
1. `Runtime` → `Change runtime type` → set **Hardware accelerator = T4 GPU**.
2. Have your `kaggle.json` API token ready (from https://www.kaggle.com/settings → *Create New API Token*). Cell 4 will prompt you to upload it.

Total wall-clock on T4 for `--epochs 35` across all 82 channels: roughly **15–30 minutes**.

## 1. Verify GPU is enabled

In [ ]:
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('No GPU. Set Runtime > Change runtime type > T4 GPU and re-run.')

## 2. Clone the repo and install deps

In [ ]:
%cd /content
![ -d multi-agent-spacecraft-anomaly-triage ] && rm -rf multi-agent-spacecraft-anomaly-triage
!git clone https://github.com/malakazlan/multi-agent-spacecraft-anomaly-triage.git
%cd /content/multi-agent-spacecraft-anomaly-triage
!pip install -q -r requirements.txt

## 3. Set up Kaggle API and download the dataset

In [ ]:
import os
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json (get it from https://www.kaggle.com/settings -> Create New API Token):')
    uploaded = files.upload()
    assert 'kaggle.json' in uploaded, 'kaggle.json not uploaded'
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json ready.')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d patrickfleith/nasa-anomaly-detection-dataset-smap-msl -p data --unzip

# Kaggle zip nests as data/data/data/{train,test}. Flatten it.
import os, shutil
nested = 'data/data/data'
if os.path.isdir(os.path.join(nested, 'train')) and os.path.isdir(os.path.join(nested, 'test')):
    for sub in ('train', 'test'):
        dst = f'data/{sub}'
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.move(os.path.join(nested, sub), dst)
    shutil.rmtree('data/data')

print('train channels:', len(os.listdir('data/train')))
print('test  channels:', len(os.listdir('data/test')))

## 4. Synthetic smoke test (fast sanity check)

In [ ]:
!python src/run.py --backend fast --limit 8

## 5. Run the real LSTM benchmark on real NASA data

This is the headline number. `--epochs 35` matches the JPL paper config; expect ~15–30 min on T4.

In [ ]:
!python src/run.py --backend lstm --epochs 35

## 6. Show final results and save them locally

In [ ]:
import json
with open('results/run.json') as f:
    run = json.load(f)

m = run['metrics']
print('=' * 60)
print(f"  backend: {run.get('backend')}    mode: {run.get('mode')}")
print(f"  precision : {m['precision']:.4f}")
print(f"  recall    : {m['recall']:.4f}")
print(f"  F1        : {m['f1']:.4f}    <- this is the number for the email")
print(f"  TP/FP/FN  : {m['TP']} / {m['FP']} / {m['FN']}")
print('-' * 60)
print(f"  JPL baseline (reference): P=0.87 R=0.80 F1=0.71")
print(f"  Decision : {run['decision']['action']} (conf {run['decision']['confidence']})")
print('=' * 60)

In [ ]:
from google.colab import files
files.download('results/run.json')